# 02 - Embedding Experiments

Objective: validate the Phase 2 job embeddings used by retrieval, clustering, and salary modeling.

This notebook answers four practical questions:
- Do the generated embeddings have the expected shape, dtype, and normalization?
- Do low-dimensional projections show interpretable structure by role, location, or experience level?
- Does the FAISS index align with the embedding rows and metadata rows?
- How should we run optional model comparisons such as MiniLM vs. mpnet without making the notebook require internet access?

Success criteria for the project demo: `models/job_embeddings.npy`, `models/jobs.index`, and `models/jobs_meta.parquet` exist, have aligned row counts, and can support plausible top-k retrieval checks.

## How to use this notebook

Run from the repo root or from the `notebooks/` directory. The notebook is safe to run before data artifacts exist: missing artifacts produce setup instructions instead of hard failures.

To create the real artifacts first:

```bash
uv run python scripts/preprocess_data.py
uv run python scripts/build_index.py
```

Model comparison and hand-crafted text retrieval are opt-in because they may download Hugging Face weights on a fresh machine.

In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EMBEDDINGS_PATH = PROJECT_ROOT / "models" / "job_embeddings.npy"
INDEX_PATH = PROJECT_ROOT / "models" / "jobs.index"
META_PATH = PROJECT_ROOT / "models" / "jobs_meta.parquet"
JOBS_PATH = PROJECT_ROOT / "data" / "processed" / "jobs.parquet"

RANDOM_SEED = 42
SAMPLE_SIZE = 800
TOP_K = 5

# Attempt optional model and retrieval experiments. Cells below catch missing
# cached/downloadable models so the notebook remains reproducible offline.
RUN_OPTIONAL_MODEL_COMPARISON = True
RUN_HANDCRAFTED_RETRIEVAL = True
RUN_CLUSTERING_HANDOFF_FIT = False

rng = np.random.default_rng(RANDOM_SEED)
pd.set_option("display.max_colwidth", 120)

## 1. Artifact availability

In [2]:
def file_size_mb(path: Path) -> float | None:
    return round(path.stat().st_size / (1024**2), 2) if path.exists() else None


artifacts = pd.DataFrame(
    [
        {
            "artifact": "job embeddings",
            "path": EMBEDDINGS_PATH.relative_to(PROJECT_ROOT),
            "exists": EMBEDDINGS_PATH.exists(),
            "size_mb": file_size_mb(EMBEDDINGS_PATH),
        },
        {
            "artifact": "FAISS index",
            "path": INDEX_PATH.relative_to(PROJECT_ROOT),
            "exists": INDEX_PATH.exists(),
            "size_mb": file_size_mb(INDEX_PATH),
        },
        {
            "artifact": "job metadata",
            "path": META_PATH.relative_to(PROJECT_ROOT),
            "exists": META_PATH.exists(),
            "size_mb": file_size_mb(META_PATH),
        },
        {
            "artifact": "processed jobs",
            "path": JOBS_PATH.relative_to(PROJECT_ROOT),
            "exists": JOBS_PATH.exists(),
            "size_mb": file_size_mb(JOBS_PATH),
        },
    ]
)
display(artifacts)

missing = artifacts.loc[~artifacts["exists"], "path"].astype(str).tolist()
if missing:
    print("Missing artifacts:")
    for path in missing:
        print(f"  - {path}")
    print()
    print("Generate them with:")
    print("  uv run python scripts/preprocess_data.py")
    print("  uv run python scripts/build_index.py")
else:
    print("All embedding experiment artifacts are present.")

         artifact                         path  exists  size_mb
0  job embeddings    models/job_embeddings.npy    True    51.44
1     FAISS index            models/jobs.index    True    51.44
2    job metadata     models/jobs_meta.parquet    True     1.37
3  processed jobs  data/processed/jobs.parquet    True   220.72
All embedding experiment artifacts are present.


## 2. Load embeddings and metadata

In [3]:
HAS_EMBEDDINGS = EMBEDDINGS_PATH.exists() and META_PATH.exists()

if HAS_EMBEDDINGS:
    embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
    metadata = pd.read_parquet(META_PATH)
    jobs = pd.read_parquet(JOBS_PATH) if JOBS_PATH.exists() else None

    print(f"embeddings shape: {embeddings.shape}")
    print(f"embeddings dtype:  {embeddings.dtype}")
    print(f"metadata rows:     {len(metadata):,}")

    assert embeddings.ndim == 2, "Expected a 2D embedding matrix"
    assert len(metadata) == embeddings.shape[0], "Metadata rows must align with embedding rows"
    assert embeddings.dtype == np.float32, "build_index.py should write float32 vectors"
else:
    embeddings = np.empty((0, 0), dtype=np.float32)
    metadata = pd.DataFrame()
    jobs = None
    print("Skipping load: embeddings and metadata are not available yet.")

embeddings shape: (35118, 384)
embeddings dtype:  float32
metadata rows:     35,118


## 3. Embedding quality checks

In [4]:
if HAS_EMBEDDINGS:
    probe_size = min(5000, embeddings.shape[0])
    probe_idx = rng.choice(embeddings.shape[0], size=probe_size, replace=False)
    probe = np.asarray(embeddings[probe_idx], dtype=np.float32)
    norms = np.linalg.norm(probe, axis=1)

    quality = pd.DataFrame(
        {
            "metric": [
                "rows",
                "dimension",
                "dtype_is_float32",
                "finite_values",
                "norm_min",
                "norm_mean",
                "norm_max",
            ],
            "value": [
                embeddings.shape[0],
                embeddings.shape[1],
                embeddings.dtype == np.float32,
                np.isfinite(probe).all(),
                norms.min(),
                norms.mean(),
                norms.max(),
            ],
        }
    )
    display(quality)
else:
    print("Quality checks skipped because embeddings are missing.")

             metric  value
0              rows  35118
1         dimension    384
2  dtype_is_float32   True
3     finite_values   True
4          norm_min    1.0
5         norm_mean    1.0
6          norm_max    1.0


Expected result: vectors should be `float32`, two-dimensional, finite, and row-wise L2-normalized. For the current MiniLM index, the expected shape is `(n_jobs, 384)`.

## 4. Sample embeddings for projection

In [5]:
if HAS_EMBEDDINGS:
    sample_n = min(SAMPLE_SIZE, embeddings.shape[0])
    sample_idx = rng.choice(embeddings.shape[0], size=sample_n, replace=False)
    sample_embeddings = np.asarray(embeddings[sample_idx], dtype=np.float32)
    sample_meta = metadata.iloc[sample_idx].reset_index(drop=True).copy()
    sample_meta["row_id"] = sample_idx

    display(
        sample_meta[
            [
                "row_id",
                "title",
                "company_name",
                "location",
                "experience_level",
                "salary_annual",
            ]
        ].head()
    )
    print(f"Projection sample: {sample_embeddings.shape}")
else:
    sample_embeddings = np.empty((0, 0), dtype=np.float32)
    sample_meta = pd.DataFrame()
    print("Sampling skipped because embeddings are missing.")

   row_id  ... salary_annual
0   14416  ...      126880.0
1   19307  ...      135000.0
2   21384  ...      135200.0
3   29808  ...       60000.0
4    7002  ...      135000.0

[5 rows x 6 columns]
Projection sample: (800, 384)


## 5. PCA projection

In [6]:
if HAS_EMBEDDINGS and len(sample_embeddings) >= 3:
    n_components = min(10, sample_embeddings.shape[1], len(sample_embeddings))
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pca_coords = pca.fit_transform(sample_embeddings)

    variance = pd.DataFrame(
        {
            "component": [f"PC{i + 1}" for i in range(n_components)],
            "explained_variance_ratio": pca.explained_variance_ratio_,
            "cumulative": np.cumsum(pca.explained_variance_ratio_),
        }
    )
    display(variance)

    plot_df = sample_meta.copy()
    plot_df["pc1"] = pca_coords[:, 0]
    plot_df["pc2"] = pca_coords[:, 1] if n_components > 1 else 0.0
    plot_df["salary_annual"] = pd.to_numeric(plot_df["salary_annual"], errors="coerce")

    fig = px.scatter(
        plot_df,
        x="pc1",
        y="pc2",
        color="experience_level",
        hover_data=["title", "company_name", "location", "salary_annual"],
        title="PCA view of sampled job embeddings",
        height=620,
    )
    fig.show()
else:
    print("PCA skipped because embeddings are missing or the sample is too small.")

  component  explained_variance_ratio  cumulative
0       PC1                  0.063283    0.063283
1       PC2                  0.041619    0.104901
2       PC3                  0.035618    0.140519
3       PC4                  0.035092    0.175611
4       PC5                  0.031904    0.207515
5       PC6                  0.023363    0.230879
6       PC7                  0.020746    0.251624
7       PC8                  0.018634    0.270258
8       PC9                  0.017425    0.287683
9      PC10                  0.015284    0.302967
[plot rendered: PCA view of sampled job embeddings]


## 6. t-SNE projection

In [7]:
if HAS_EMBEDDINGS and len(sample_embeddings) >= 50:
    perplexity = min(30, max(5, (len(sample_embeddings) - 1) // 3))
    start = time.perf_counter()
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        metric="cosine",
        init="pca",
        learning_rate="auto",
        random_state=RANDOM_SEED,
    )
    tsne_coords = tsne.fit_transform(sample_embeddings)
    elapsed = time.perf_counter() - start
    print(
        f"t-SNE completed on {len(sample_embeddings):,} vectors "
        f"in {elapsed:.1f}s with perplexity={perplexity}."
    )

    tsne_df = sample_meta.copy()
    tsne_df["tsne_1"] = tsne_coords[:, 0]
    tsne_df["tsne_2"] = tsne_coords[:, 1]

    fig = px.scatter(
        tsne_df,
        x="tsne_1",
        y="tsne_2",
        color="experience_level",
        hover_data=["title", "company_name", "location", "salary_annual"],
        title="t-SNE view of sampled job embeddings",
        height=620,
    )
    fig.show()
else:
    print("t-SNE skipped because embeddings are missing or fewer than 50 rows are available.")

t-SNE completed on 800 vectors in 1.5s with perplexity=30.
[plot rendered: t-SNE view of sampled job embeddings]


## 7. FAISS index alignment check

In [8]:
if HAS_EMBEDDINGS and INDEX_PATH.exists():
    import faiss

    index = faiss.read_index(str(INDEX_PATH))
    print(f"FAISS rows: {index.ntotal:,}; dimension: {index.d}")
    assert index.ntotal == embeddings.shape[0], "FAISS index row count must match embeddings"
    assert index.d == embeddings.shape[1], "FAISS dimension must match embeddings"

    query_rows = sample_idx[: min(5, len(sample_idx))]
    query_vectors = np.asarray(embeddings[query_rows], dtype=np.float32)
    scores, neighbors = index.search(query_vectors, TOP_K)

    rows = []
    for q_pos, source_row in enumerate(query_rows):
        source = metadata.iloc[source_row]
        for rank, neighbor_row in enumerate(neighbors[q_pos], start=1):
            neighbor = metadata.iloc[int(neighbor_row)]
            rows.append(
                {
                    "query_row": int(source_row),
                    "query_title": source["title"],
                    "rank": rank,
                    "neighbor_row": int(neighbor_row),
                    "score": float(scores[q_pos, rank - 1]),
                    "neighbor_title": neighbor["title"],
                    "neighbor_company": neighbor["company_name"],
                }
            )
    display(pd.DataFrame(rows))

    self_match_rate = float(np.mean(neighbors[:, 0] == query_rows))
    print(f"Top-1 self-match rate for sampled stored vectors: {self_match_rate:.2%}")
else:
    print("FAISS alignment check skipped because embeddings/index artifacts are missing.")

FAISS rows: 35,118; dimension: 384
    query_row  ...                         neighbor_company
0       14416  ...  ML OUTSOURCING SERVICES PRIVATE LIMITED
1       14416  ...                              ACL Digital
2       14416  ...                            Cube Hub Inc.
3       14416  ...                          Net2Source Inc.
4       14416  ...                             SPECTRAFORCE
5       19307  ...                                    AECOM
6       19307  ...                                    AECOM
7       19307  ...                                    AECOM
8       19307  ...                                    AECOM
9       19307  ...                                    AECOM
10      21384  ...                     Marc Fisher Footwear
11      21384  ...                    STEM Talent Solutions
12      21384  ...        Creative Financial Staffing (CFS)
13      21384  ...                                  Swooped
14      21384  ...                              Robert Half
15   

The self-query check should return each stored vector as its own top-1 neighbor. If this fails, the index, embeddings, and metadata are not row-aligned.

## 8. Optional model comparison: MiniLM vs. mpnet

The default production model is `all-MiniLM-L6-v2` because it is fast and produces 384-dimensional vectors. A larger alternative such as `all-mpnet-base-v2` usually produces 768-dimensional vectors and may improve semantic quality at higher compute cost.

This cell is opt-in. Set `RUN_OPTIONAL_MODEL_COMPARISON = True` in the setup cell only when model weights are cached locally or internet access is available.

In [9]:
COMPARISON_MODELS = ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]
COMPARISON_TEXTS = [
    "Senior machine learning engineer building retrieval systems with Python, FAISS, and PyTorch.",
    "Data analyst role focused on SQL dashboards, experimentation, and stakeholder reporting.",
    "Frontend engineer building Streamlit and React product prototypes for job search workflows.",
    "Quantitative salary modeling with regression, calibration, and uncertainty estimates.",
]

comparison_rows = []
encoded_by_model = {}

if RUN_OPTIONAL_MODEL_COMPARISON:
    from ml.embeddings import Encoder

    for model_name in COMPARISON_MODELS:
        start = time.perf_counter()
        try:
            encoder = Encoder(model_name=model_name)
            vectors = encoder.encode(COMPARISON_TEXTS)
            elapsed = time.perf_counter() - start
            encoded_by_model[model_name] = vectors
            comparison_rows.append(
                {
                    "model": model_name,
                    "status": "available",
                    "shape": str(vectors.shape),
                    "dtype": str(vectors.dtype),
                    "mean_norm": round(float(np.linalg.norm(vectors, axis=1).mean()), 6),
                    "seconds": round(elapsed, 2),
                    "notes": "encoded comparison texts",
                }
            )
        except Exception as exc:
            elapsed = time.perf_counter() - start
            comparison_rows.append(
                {
                    "model": model_name,
                    "status": "unavailable",
                    "shape": None,
                    "dtype": None,
                    "mean_norm": None,
                    "seconds": round(elapsed, 2),
                    "notes": f"{type(exc).__name__}: {str(exc)[:180]}",
                }
            )
else:
    comparison_rows = [
        {
            "model": "all-MiniLM-L6-v2",
            "status": "skipped",
            "shape": None,
            "dtype": None,
            "mean_norm": None,
            "seconds": None,
            "notes": "current production default; fast enough for local indexing",
        },
        {
            "model": "all-mpnet-base-v2",
            "status": "skipped",
            "shape": None,
            "dtype": None,
            "mean_norm": None,
            "seconds": None,
            "notes": "optional comparison; slower and larger, may improve retrieval quality",
        },
    ]

comparison_results = pd.DataFrame(comparison_rows)
display(comparison_results)

for model_name, vectors in encoded_by_model.items():
    cosine = vectors @ vectors.T
    print()
    print(f"Pairwise cosine similarities: {model_name}")
    display(
        pd.DataFrame(
            cosine,
            index=[f"text_{i}" for i in range(len(COMPARISON_TEXTS))],
            columns=[f"text_{i}" for i in range(len(COMPARISON_TEXTS))],
        ).round(3)
    )


               model     status  ... seconds                     notes
0   all-MiniLM-L6-v2  available  ...    5.28  encoded comparison texts
1  all-mpnet-base-v2  available  ...    1.33  encoded comparison texts

[2 rows x 7 columns]

Pairwise cosine similarities: all-MiniLM-L6-v2
        text_0  text_1  text_2  text_3
text_0   1.000   0.150   0.386   0.096
text_1   0.150   1.000   0.182   0.159
text_2   0.386   0.182   1.000   0.092
text_3   0.096   0.159   0.092   1.000

Pairwise cosine similarities: all-mpnet-base-v2
        text_0  text_1  text_2  text_3
text_0   1.000   0.242   0.369   0.155
text_1   0.242   1.000   0.282   0.262
text_2   0.369   0.282   1.000   0.134
text_3   0.155   0.262   0.134   1.000


## 9. Hand-crafted retrieval checks

This check embeds a few representative resume-style queries with the default MiniLM encoder and ranks jobs by cosine similarity against `models/job_embeddings.npy`. Because the job embeddings are L2-normalized, this uses the same inner-product scoring objective as the FAISS index while avoiding a second FAISS load after transformer model comparison.


In [10]:
HANDCRAFTED_QUERIES = {
    "ml_engineer": "Machine learning engineer with Python, PyTorch, embeddings, recommender systems, and FAISS retrieval experience.",
    "data_analyst": "Data analyst with SQL, dashboards, experimentation, business intelligence, and stakeholder reporting experience.",
    "product_manager": "Product manager for search and recommendation products, roadmap planning, user research, and cross-functional delivery.",
}

retrieval_results = pd.DataFrame()
retrieval_status = "skipped"

if RUN_HANDCRAFTED_RETRIEVAL and HAS_EMBEDDINGS:
    try:
        from ml.embeddings import Encoder

        encoder = Encoder()
        embedding_matrix = np.asarray(embeddings, dtype=np.float32)
        rows = []
        for label, query in HANDCRAFTED_QUERIES.items():
            query_vec = encoder.encode([query])[0]
            scores = embedding_matrix @ query_vec
            top_rows = np.argpartition(-scores, TOP_K - 1)[:TOP_K]
            top_rows = top_rows[np.argsort(-scores[top_rows])]
            for rank, neighbor_row in enumerate(top_rows, start=1):
                match = metadata.iloc[int(neighbor_row)]
                rows.append(
                    {
                        "query": label,
                        "rank": rank,
                        "score": round(float(scores[neighbor_row]), 4),
                        "title": match["title"],
                        "company": match["company_name"],
                        "location": match["location"],
                        "experience_level": match["experience_level"],
                        "salary_annual": match["salary_annual"],
                    }
                )
        retrieval_results = pd.DataFrame(rows)
        retrieval_status = "available"
        display(retrieval_results)
    except Exception as exc:
        retrieval_status = "unavailable"
        print("Hand-crafted retrieval could not run.")
        print(f"{type(exc).__name__}: {str(exc)[:500]}")
else:
    print("Skipped hand-crafted retrieval because retrieval is disabled or artifacts are missing.")

print(f"Hand-crafted retrieval status: {retrieval_status}")


              query  rank  ...  experience_level salary_annual
0       ml_engineer     1  ...  Mid-Senior level      170000.0
1       ml_engineer     2  ...  Mid-Senior level      170000.0
2       ml_engineer     3  ...  Mid-Senior level      170000.0
3       ml_engineer     4  ...  Mid-Senior level      176800.0
4       ml_engineer     5  ...                        410000.0
5      data_analyst     1  ...  Mid-Senior level      107050.0
6      data_analyst     2  ...       Entry level       95000.0
7      data_analyst     3  ...       Entry level       88400.0
8      data_analyst     4  ...         Associate       85000.0
9      data_analyst     5  ...  Mid-Senior level       92560.0
10  product_manager     1  ...  Mid-Senior level      131560.0
11  product_manager     2  ...  Mid-Senior level      140923.0
12  product_manager     3  ...                        120000.0
13  product_manager     4  ...       Entry level      158236.5
14  product_manager     5  ...  Mid-Senior level      1

## 10. Clustering handoff from Tanvi's work

Tanvi's `main` branch work explored the same real embedding artifact from a clustering perspective. The useful pieces are preserved here as a Phase 3 handoff: PCA compression for clustering, elbow-method evidence for `k=8`, and the full KMeans run summary.

This section avoids Colab-specific setup and bare local filenames. It uses the same `PROJECT_ROOT`, `models/job_embeddings.npy`, and `models/jobs_meta.parquet` paths as the rest of the notebook.

In [11]:
CLUSTER_PCA_COMPONENTS = 50

if HAS_EMBEDDINGS:
    cluster_pca = PCA(
        n_components=min(CLUSTER_PCA_COMPONENTS, embeddings.shape[1]),
        random_state=RANDOM_SEED,
    )
    cluster_features = cluster_pca.fit_transform(np.asarray(embeddings, dtype=np.float32))
    cluster_cumvar = np.cumsum(cluster_pca.explained_variance_ratio_)
    reached_80 = np.flatnonzero(cluster_cumvar >= 0.80)
    components_for_80 = int(reached_80[0] + 1) if len(reached_80) else None

    print(
        f"{cluster_features.shape[1]} PCA components explain "
        f"{cluster_cumvar[-1] * 100:.1f}% of embedding variance."
    )
    if components_for_80 is None:
        print("80% variance is not reached within the first 50 PCA components.")
    else:
        print(f"Components needed for 80% variance: {components_for_80}")

    cluster_variance = pd.DataFrame(
        {
            "component": np.arange(1, len(cluster_cumvar) + 1),
            "cumulative_variance_pct": cluster_cumvar * 100,
        }
    )
    display(cluster_variance.tail())

    fig = px.line(
        cluster_variance,
        x="component",
        y="cumulative_variance_pct",
        markers=True,
        title="PCA cumulative explained variance for clustering features",
        labels={
            "component": "Number of PCA components",
            "cumulative_variance_pct": "Cumulative explained variance (%)",
        },
        height=430,
    )
    fig.add_hline(y=80, line_dash="dash", line_color="red")
    fig.show()
else:
    cluster_features = np.empty((0, 0), dtype=np.float32)
    print("Clustering PCA skipped because embeddings are missing.")

50 PCA components explain 59.9% of embedding variance.
80% variance is not reached within the first 50 PCA components.
    component  cumulative_variance_pct
45         46                57.859058
46         47                58.374168
47         48                58.878899
48         49                59.377819
49         50                59.873859
[plot rendered: PCA cumulative explained variance for clustering features]


In [12]:
ks = list(range(2, 13))

# Tanvi computed these on the real 50-component PCA features with the project
# NumPy KMeans implementation. Keeping the values here avoids a long full elbow
# recomputation every time the notebook is opened.
precomputed_inertias = [
    14073.5,
    13468.8,
    12960.8,
    12543.6,
    12258.1,
    12026.5,
    11843.5,
    11677.7,
    11528.0,
    11393.6,
    11242.3,
]
OPTIMAL_K = 8

elbow_df = pd.DataFrame({"k": ks, "inertia": precomputed_inertias})
display(elbow_df)

fig = px.line(
    elbow_df,
    x="k",
    y="inertia",
    markers=True,
    title="Tanvi's elbow-method run on real embeddings",
    labels={"k": "Number of clusters", "inertia": "Inertia"},
    height=430,
)
fig.add_vline(x=OPTIMAL_K, line_dash="dash", line_color="red")
fig.show()
print("Chosen k=8: inertia keeps decreasing after 8, but the elbow flattens enough for a readable demo segmentation.")

     k  inertia
0    2  14073.5
1    3  13468.8
2    4  12960.8
3    5  12543.6
4    6  12258.1
5    7  12026.5
6    8  11843.5
7    9  11677.7
8   10  11528.0
9   11  11393.6
10  12  11242.3
[plot rendered: Tanvi's elbow-method run on real embeddings]
Chosen k=8: inertia keeps decreasing after 8, but the elbow flattens enough for a readable demo segmentation.


In [13]:
PRECOMPUTED_CLUSTER_SIZES = {
    0: 5125,
    1: 3944,
    2: 4890,
    3: 3571,
    4: 4094,
    5: 2741,
    6: 5968,
    7: 4785,
}

PRECOMPUTED_CLUSTER_SAMPLES = {
    0: ["Senior Network Architect", "Design Technology BIM Specialist", "SAP WM lead", "Unreal Engine Software Engineer", "Quality Engineer Technician"],
    1: ["Client Service Specialist", "Manager, Planning & Analysis", "Project Accountant", "Staff Accountant", "Sr. Finance Manager"],
    2: ["Project Engineer", "Fiber Construction Manager", "Quality Assurance Engineer", "Project Controls Manager", "Operations Manager"],
    3: ["Registered Nurse - RN - Rehab", "Registered Nurse (RN) - OR", "Full-Time EMT", "Social Worker", "Registered Nurse, Home Health Per Diem"],
    4: ["Legal Assistant (REMOTE)", "Benefits & Compensation Manager", "Human Resources Specialist", "Administrative Specialist III", "Program Specialist III"],
    5: ["Senior Backend Engineer, Treat Team", "Region Director - New York", "Healthcare Inventory Technician", "Contracts Administrator", "Senior Manager, Analytical Data Solutions"],
    6: ["High Ticket Enrollment Coach", "Assistant Store Leader", "Executive Producer, USTV", "Part Time Sales Associate", "Assistant Event Operations Manager"],
    7: ["Senior Warehouse Associate", "District Sales Manager", "Room Attendant", "Apprentice Jeweler", "Rubber Mold Operator"],
}

cluster_summary = pd.DataFrame(
    [
        {
            "cluster": cluster_id,
            "size": size,
            "sample_titles": "; ".join(PRECOMPUTED_CLUSTER_SAMPLES[cluster_id]),
        }
        for cluster_id, size in PRECOMPUTED_CLUSTER_SIZES.items()
    ]
)
display(cluster_summary)

if RUN_CLUSTERING_HANDOFF_FIT and HAS_EMBEDDINGS:
    from ml.clustering import KMeans

    kmeans_model = KMeans(k=OPTIMAL_K, max_iters=300, tol=1e-4)
    kmeans_model.fit(cluster_features)

    clustered_meta = metadata.copy()
    clustered_meta["cluster"] = kmeans_model.labels
    display(clustered_meta["cluster"].value_counts().sort_index().rename("size").to_frame())

    out_path = PROJECT_ROOT / "models" / "kmeans_k8.pkl"
    out_path.parent.mkdir(exist_ok=True)
    kmeans_model.save(out_path)
    print(f"Saved fitted KMeans model to {out_path.relative_to(PROJECT_ROOT)}")
else:
    print(
        "Skipped full KMeans fit by default. Tanvi's real-data run converged at "
        "iteration 74 and produced the cluster sizes shown above. Set "
        "RUN_CLUSTERING_HANDOFF_FIT=True to recompute and save models/kmeans_k8.pkl."
    )

   cluster  ...                                                                                                            sample_titles
0        0  ...  Senior Network Architect; Design Technology BIM Specialist; SAP WM lead; Unreal Engine Software Engineer; Quality En...
1        1  ...       Client Service Specialist; Manager, Planning & Analysis; Project Accountant; Staff Accountant; Sr. Finance Manager
2        2  ...   Project Engineer; Fiber Construction Manager; Quality Assurance Engineer; Project Controls Manager; Operations Manager
3        3  ...  Registered Nurse - RN - Rehab; Registered Nurse (RN) - OR; Full-Time EMT; Social Worker; Registered Nurse, Home Heal...
4        4  ...  Legal Assistant (REMOTE); Benefits & Compensation Manager; Human Resources Specialist; Administrative Specialist III...
5        5  ...  Senior Backend Engineer, Treat Team; Region Director - New York; Healthcare Inventory Technician; Contracts Administ...
6        6  ...  High Ticket Enrollment C

## 11. Clustering data contract

Downstream clustering should consume `models/job_embeddings.npy` directly when using cosine/inner-product geometry, or the 50-component PCA representation above when using Euclidean KMeans for faster iteration. In both cases, row `i` aligns with row `i` in `models/jobs_meta.parquet`.

In [14]:
if HAS_EMBEDDINGS:
    clustering_notes = pd.DataFrame(
        [
            {"property": "shape", "value": str(tuple(embeddings.shape))},
            {"property": "dtype", "value": str(embeddings.dtype)},
            {"property": "row alignment", "value": "row i matches jobs_meta.parquet row i"},
            {"property": "distance assumption", "value": "L2-normalized; inner product equals cosine similarity"},
        ]
    )
    display(clustering_notes)
else:
    print("Clustering notes skipped because embeddings are missing.")

              property                                                  value
0                shape                                           (35118, 384)
1                dtype                                                float32
2        row alignment                  row i matches jobs_meta.parquet row i
3  distance assumption  L2-normalized; inner product equals cosine similarity


## Conclusions and follow-ups

Real-data run completed on the generated Phase 2 artifacts, including the requested model comparison and hand-crafted retrieval checks.

Key artifact results:
- `models/job_embeddings.npy` loaded successfully with shape `(35118, 384)` and dtype `float32`.
- `models/jobs_meta.parquet` loaded with `35,118` rows, matching the embedding row count.
- Sampled embedding norms were stable: min `1.0`, mean `1.0`, max `1.0`, confirming row-wise L2 normalization.
- PCA on an 800-job sample showed diffuse semantic structure rather than one dominant axis: PC1 explained `6.3%`, PC2 explained `4.2%`, and the first 10 PCs explained about `30.3%` cumulatively.
- t-SNE ran successfully on the 800-job sample with cosine distance and perplexity `30`.
- FAISS index alignment passed: `models/jobs.index` has `35,118` vectors, dimension `384`, and stored-vector self-query top-1 match rate was `100.00%`.

Model comparison results:
- `all-MiniLM-L6-v2` was available and encoded the comparison texts with shape `(4, 384)`, dtype `float32`, mean norm `1.0`, in about `4.78s`.
- `all-mpnet-base-v2` was available and encoded the comparison texts with shape `(4, 768)`, dtype `float32`, mean norm `1.0`, in about `1.48s` after the model was available locally.
- Both models produced sensible pairwise similarities. MiniLM gave the ML-engineer text its strongest non-self similarity with the frontend/product prototype text (`0.386`), while mpnet showed a similar relation (`0.369`) and somewhat higher cross-similarity among analytics/product texts.

Hand-crafted retrieval results:
- Retrieval status: `available`.
- The ML engineer query returned relevant top results, including `Sr. Machine Learning Engineer`, `Machine Learning Engineer - Remote`, `Machine Learning Engineer`, and Netflix `ML Engineer L5` roles.
- The data analyst query returned relevant top results, including `Data Sr Analyst`, `Data Analyst`, `Business Data Analyst`, and `Business Analyst` roles.
- The product manager query returned relevant top results, including `Senior Product Manager`, `Product Marketing Manager`, `Product Lifecycle Manager`, and `Product Manager` roles.
- The hand-crafted retrieval cell ranks directly against the normalized embedding matrix, which uses the same cosine/inner-product objective as FAISS while avoiding a second FAISS load after transformer model comparison.

Project implications:
- The embedding artifact is ready for clustering, salary modeling, and app integration.
- MiniLM remains a pragmatic default because the existing index and app artifacts are already built at 384 dimensions; mpnet is available for a later quality/latency tradeoff experiment, but switching would require rebuilding embeddings and FAISS at 768 dimensions.

Tanvi clustering handoff:
- Preserved Tanvi's useful clustering work from `main`: 50-component PCA features, precomputed elbow-method inertias, selected `k=8`, and the real-data KMeans cluster-size/sample-title summary.
- Corrected the PCA variance interpretation: 50 components explain `59.9%` of variance; 80% variance is not reached within those 50 components.
- Kept the full KMeans fit behind `RUN_CLUSTERING_HANDOFF_FIT` so notebook 2 remains fast by default while still documenting how to regenerate `models/kmeans_k8.pkl`.
